# Reset, quit

In [6]:
import pygame
import random
import numpy as np
import sys
from collections import deque
import math
import atexit
import time

# Initialize pygame
pygame.init()

# Constants
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
CELL_SIZE = 40
GRID_WIDTH = 20
GRID_HEIGHT = 15
PANEL_WIDTH = 400
FPS = 60

# Colors
BACKGROUND = (30, 30, 50)
GRID_LINES = (60, 60, 80)
WALL_COLOR = (50, 50, 70)
PATH_COLOR = (35, 35, 55)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
PANEL_BG = (25, 25, 40)
BUTTON_COLOR = (70, 130, 200)
BUTTON_HOVER = (90, 150, 220)
Q_VALUE_COLORS = [
    (180, 60, 60),    # Low values (red)
    (220, 150, 60),   # Medium values (orange)
    (60, 180, 80)     # High values (green)
]

# Create screen
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Reinforcement Learning Maze Environment")
clock = pygame.time.Clock()

# Fonts
font_large = pygame.font.SysFont("Arial", 36, bold=True)
font_medium = pygame.font.SysFont("Arial", 24)
font_small = pygame.font.SysFont("Arial", 18)

# Register cleanup function
def cleanup():
    pygame.quit()

atexit.register(cleanup)

class Maze:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)  # 0 = path, 1 = wall
        self.start_pos = (1, 1)
        self.goal_pos = (width - 2, height - 2)
        self.hazards = []
        self.generate_maze()
        
    def generate_maze(self):
        # Initialize with walls
        self.grid.fill(1)
        
        # Create a depth-first search maze
        stack = [self.start_pos]
        self.grid[self.start_pos[1]][self.start_pos[0]] = 0
        
        # Possible directions
        directions = [(0, -2), (2, 0), (0, 2), (-2, 0)]  # Up, right, down, left
        
        while stack:
            current = stack[-1]
            x, y = current
            
            # Find unvisited neighbors
            neighbors = []
            for dx, dy in directions:
                nx, ny = x + dx, y + dy
                if 0 < nx < self.width - 1 and 0 < ny < self.height - 1 and self.grid[ny][nx] == 1:
                    neighbors.append((nx, ny, dx, dy))
            
            if neighbors:
                # Choose a random neighbor
                nx, ny, dx, dy = random.choice(neighbors)
                
                # Remove wall between current and neighbor
                self.grid[y + dy//2][x + dx//2] = 0
                self.grid[ny][nx] = 0
                
                # Add to stack
                stack.append((nx, ny))
            else:
                # Backtrack
                stack.pop()
        
        # Ensure there's a path to the goal
        self.create_path_to_goal()
        
        # Add hazards (10% of path cells)
        path_cells = [(x, y) for y in range(self.height) for x in range(self.width) 
                      if self.grid[y][x] == 0 and (x, y) != self.start_pos and (x, y) != self.goal_pos]
        
        num_hazards = max(1, int(len(path_cells) * 0.1))
        self.hazards = random.sample(path_cells, num_hazards)
    
    def create_path_to_goal(self):
        # Use BFS to find a path from start to goal
        queue = deque([self.start_pos])
        visited = {self.start_pos: None}
        
        while queue:
            x, y = queue.popleft()
            
            if (x, y) == self.goal_pos:
                break
                
            for dx, dy in [(0, -1), (1, 0), (0, 1), (-1, 0)]:
                nx, ny = x + dx, y + dy
                if (0 <= nx < self.width and 0 <= ny < self.height and 
                    self.grid[ny][nx] == 0 and (nx, ny) not in visited):
                    queue.append((nx, ny))
                    visited[(nx, ny)] = (x, y)
        
        # If no path found, create one
        if self.goal_pos not in visited:
            x, y = self.goal_pos
            path = []
            
            # Create a path backwards from goal to start
            while (x, y) != self.start_pos:
                path.append((x, y))
                # Move toward start (simplified)
                if x > self.start_pos[0]:
                    x -= 1
                elif x < self.start_pos[0]:
                    x += 1
                elif y > self.start_pos[1]:
                    y -= 1
                elif y < self.start_pos[1]:
                    y += 1
                
                # Remove wall at this position
                if 0 <= x < self.width and 0 <= y < self.height:
                    self.grid[y][x] = 0
            
            # Add the path to the maze
            for pos in path:
                self.grid[pos[1]][pos[0]] = 0
    
    def is_wall(self, x, y):
        return self.grid[y][x] == 1
    
    def is_hazard(self, x, y):
        return (x, y) in self.hazards

class Player:
    def __init__(self, maze):
        self.maze = maze
        self.reset()
        self.lives = 30
        self.speed = 0.2
        self.radius = CELL_SIZE // 3
        self.trail = []
        
    def reset(self):
        self.x, self.y = self.maze.start_pos
        self.px, self.py = self.x, self.y
        self.target_x, self.target_y = self.x, self.y
        self.moving = False
        self.trail = []
        
    def move(self, dx, dy):
        if self.moving:
            return
            
        new_x, new_y = self.x + dx, self.y + dy
        
        # Check boundaries
        if (0 <= new_x < self.maze.width and 0 <= new_y < self.maze.height and 
            not self.maze.is_wall(new_x, new_y)):
            self.px, self.py = self.x, self.y
            self.target_x, self.target_y = new_x, new_y
            self.moving = True
            
            # Add to trail with current timestamp
            self.trail.append((self.x, self.y, time.time()))
    
    def update(self):
        if self.moving:
            # Move toward target
            dx = self.target_x - self.x
            dy = self.target_y - self.y
            dist = math.sqrt(dx*dx + dy*dy)
            
            if dist < self.speed:
                self.x, self.y = self.target_x, self.target_y
                self.moving = False
            else:
                self.x += dx * self.speed
                self.y += dy * self.speed
        
        # Remove trail points older than 1 second
        current_time = time.time()
        self.trail = [(x, y, t) for (x, y, t) in self.trail if current_time - t < 1.0]
    
    def get_position(self):
        return (int(self.x), int(self.y))
    
    def lose_life(self):
        self.lives -= 1
        self.reset()
        return self.lives <= 0
    
    def draw(self, screen, offset_x, offset_y):
        # Draw trail
        current_time = time.time()
        for i, (tx, ty, t) in enumerate(self.trail):
            # Calculate alpha based on remaining time (fade out over 1 second)
            alpha = int(255 * (1 - (current_time - t)))
            if alpha <= 0:
                continue
                
            # Create a surface with per-pixel alpha
            trail_surface = pygame.Surface((CELL_SIZE, CELL_SIZE), pygame.SRCALPHA)
            radius = int(self.radius * (1 - (current_time - t)))
            pygame.draw.circle(trail_surface, (70, 130, 180, alpha), 
                              (CELL_SIZE // 2, CELL_SIZE // 2), 
                              radius)
            screen.blit(trail_surface, (offset_x + tx * CELL_SIZE, offset_y + ty * CELL_SIZE))
        
        # Draw player
        pygame.draw.circle(screen, PLAYER_COLOR, 
                          (offset_x + (self.x + 0.5) * CELL_SIZE, offset_y + (self.y + 0.5) * CELL_SIZE), 
                          self.radius)
        
        # Draw player direction indicator
        if self.moving:
            dx = self.target_x - self.px
            dy = self.target_y - self.py
            if dx != 0 or dy != 0:
                angle = math.atan2(dy, dx)
                end_x = (self.x + 0.5 + 0.6 * math.cos(angle)) * CELL_SIZE
                end_y = (self.y + 0.5 + 0.6 * math.sin(angle)) * CELL_SIZE
                pygame.draw.line(screen, (255, 255, 255), 
                                (offset_x + (self.x + 0.5) * CELL_SIZE, offset_y + (self.y + 0.5) * CELL_SIZE),
                                (offset_x + end_x, offset_y + end_y), 3)

class QLearningAgent:
    def __init__(self, maze, player):
        self.maze = maze
        self.player = player
        self.q_table = {}
        self.learning_rate = 0.1
        self.discount_factor = 0.95
        self.exploration_rate = 0.3
        self.state = None
        self.action = None
        
    def get_state(self):
        return self.player.get_position()
    
    def get_q_value(self, state, action):
        if state not in self.q_table:
            self.q_table[state] = [0.0, 0.0, 0.0, 0.0]  # Up, Right, Down, Left
        return self.q_table[state][action]
    
    def choose_action(self, state):
        # Exploration vs exploitation
        if random.random() < self.exploration_rate:
            return random.randint(0, 3)  # Random action
        else:
            # Choose best action
            q_values = [self.get_q_value(state, a) for a in range(4)]
            max_q = max(q_values)
            # If multiple actions have the same max q value, choose randomly among them
            actions = [a for a, q in enumerate(q_values) if q == max_q]
            return random.choice(actions)
    
    def update_q_value(self, state, action, reward, next_state):
        current_q = self.get_q_value(state, action)
        next_max_q = max([self.get_q_value(next_state, a) for a in range(4)])
        
        # Q-learning formula
        new_q = current_q + self.learning_rate * (reward + self.discount_factor * next_max_q - current_q)
        self.q_table[state][action] = new_q
    
    def start_episode(self):
        self.state = self.get_state()
        self.action = self.choose_action(self.state)
        return self.action
    
    def step(self, reward, next_state):
        # Update Q-value for previous state-action pair
        self.update_q_value(self.state, self.action, reward, next_state)
        
        # Choose next action
        self.state = next_state
        self.action = self.choose_action(self.state)
        return self.action
    
    def get_q_values(self):
        return self.q_table

class Button:
    def __init__(self, x, y, width, height, text, action=None):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.action = action
        self.hovered = False
        
    def draw(self, screen):
        color = BUTTON_HOVER if self.hovered else BUTTON_COLOR
        pygame.draw.rect(screen, color, self.rect, border_radius=8)
        pygame.draw.rect(screen, (100, 160, 230), self.rect, 2, border_radius=8)
        
        text_surf = font_medium.render(self.text, True, TEXT_COLOR)
        text_rect = text_surf.get_rect(center=self.rect.center)
        screen.blit(text_surf, text_rect)
        
    def check_hover(self, pos):
        self.hovered = self.rect.collidepoint(pos)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered and self.action:
                return self.action()
        return None

class Game:
    def __init__(self):
        self.maze = Maze(GRID_WIDTH, GRID_HEIGHT)
        self.player = Player(self.maze)
        self.agent = QLearningAgent(self.maze, self.player)
        self.game_state = "PLAYING"  # PLAYING, DIED, WON
        self.offset_x = 50
        self.offset_y = 50
        self.manual_control = True
        self.episode_count = 0
        self.steps = 0
        self.episode_reward = 0
        self.total_reward = 0
        self.convergence_data = []
        self.running = True
        self.start_time = time.time()
        
        # Create buttons
        button_width = 150
        button_height = 40
        button_spacing = 20
        panel_center = SCREEN_WIDTH - PANEL_WIDTH // 2
        self.buttons = [
            Button(panel_center - button_width//2, 500, button_width, button_height, "Manual Control", self.set_manual),
            Button(panel_center - button_width//2, 500 + button_height + button_spacing, button_width, button_height, "Auto Training", self.set_auto),
            Button(panel_center - button_width//2, 500 + 2*(button_height + button_spacing), button_width, button_height, "Reset Game", self.reset_game),
            Button(panel_center - button_width//2, 500 + 3*(button_height + button_spacing), button_width, button_height, "Quit", self.quit_game)
        ]
    
    def set_manual(self):
        self.manual_control = True
        return True
    
    def set_auto(self):
        self.manual_control = False
        return True
    
    def reset_game(self):
        self.maze = Maze(GRID_WIDTH, GRID_HEIGHT)
        self.player = Player(self.maze)
        self.agent = QLearningAgent(self.maze, self.player)
        self.game_state = "PLAYING"
        self.episode_count = 0
        self.steps = 0
        self.episode_reward = 0
        self.total_reward = 0
        self.convergence_data = []
        self.start_time = time.time()
        return True
    
    def quit_game(self):
        self.running = False
        return True
    
    def get_elapsed_time(self):
        return int(time.time() - self.start_time)
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.running = False
                return False
                
            # Handle button events
            for button in self.buttons:
                button.handle_event(event)
            
            # Handle keyboard events
            if event.type == pygame.KEYDOWN and self.game_state == "PLAYING":
                if event.key == pygame.K_UP and self.manual_control:
                    self.player.move(0, -1)
                elif event.key == pygame.K_RIGHT and self.manual_control:
                    self.player.move(1, 0)
                elif event.key == pygame.K_DOWN and self.manual_control:
                    self.player.move(0, 1)
                elif event.key == pygame.K_LEFT and self.manual_control:
                    self.player.move(-1, 0)
                elif event.key == pygame.K_r:
                    self.player.reset()
                elif event.key == pygame.K_SPACE:
                    self.manual_control = not self.manual_control
                elif event.key == pygame.K_ESCAPE:
                    self.running = False
                    return False
                    
            # Handle mouse events for buttons
            if event.type == pygame.MOUSEMOTION:
                for button in self.buttons:
                    button.check_hover(event.pos)
        
        return True
    
    def update(self):
        if self.game_state != "PLAYING":
            return
            
        self.player.update()
        player_pos = self.player.get_position()
        
        # Check for hazard collision
        if self.maze.is_hazard(player_pos[0], player_pos[1]):
            dead = self.player.lose_life()
            if dead:
                self.game_state = "DIED"
            self.episode_reward -= 50
        
        # Check for goal reached
        if player_pos == self.maze.goal_pos:
            self.player.reset()
            self.episode_count += 1
            self.episode_reward += 100
            self.convergence_data.append(self.steps)
            self.steps = 0
            self.game_state = "WON"
        
        # Auto control (Q-learning)
        if not self.manual_control and not self.player.moving:
            # Get current state
            current_state = self.agent.get_state()
            
            # Choose and perform action
            if self.agent.state is None:
                action = self.agent.start_episode()
            else:
                # Calculate reward
                reward = -0.1  # Small penalty for each step
                self.episode_reward += reward
                self.total_reward += reward
                
                # Update Q-value and get next action
                action = self.agent.step(reward, current_state)
            
            # Perform action
            if action == 0:  # Up
                self.player.move(0, -1)
            elif action == 1:  # Right
                self.player.move(1, 0)
            elif action == 2:  # Down
                self.player.move(0, 1)
            elif action == 3:  # Left
                self.player.move(-1, 0)
            
            self.steps += 1
    
    def draw(self):
        screen.fill(BACKGROUND)
        
        # Draw maze and player
        self.draw_maze()
        self.player.draw(screen, self.offset_x, self.offset_y)
        
        # Draw right panel
        self.draw_panel()
        
        # Draw game state messages
        if self.game_state == "DIED":
            self.draw_message("YOU DIED!", HAZARD_COLOR)
        elif self.game_state == "WON":
            self.draw_message("LEVEL COMPLETE!", GOAL_COLOR)
        
        pygame.display.flip()
    
    def draw_maze(self):
        # Draw maze grid
        for y in range(self.maze.height):
            for x in range(self.maze.width):
                rect = pygame.Rect(
                    self.offset_x + x * CELL_SIZE,
                    self.offset_y + y * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                if self.maze.is_wall(x, y):
                    pygame.draw.rect(screen, WALL_COLOR, rect)
                else:
                    pygame.draw.rect(screen, PATH_COLOR, rect)
                
                # Draw grid lines
                pygame.draw.rect(screen, GRID_LINES, rect, 1)
        
        # Draw start and goal
        start_rect = pygame.Rect(
            self.offset_x + self.maze.start_pos[0] * CELL_SIZE,
            self.offset_y + self.maze.start_pos[1] * CELL_SIZE,
            CELL_SIZE, CELL_SIZE
        )
        pygame.draw.rect(screen, (70, 100, 150), start_rect)
        
        goal_rect = pygame.Rect(
            self.offset_x + self.maze.goal_pos[0] * CELL_SIZE,
            self.offset_y + self.maze.goal_pos[1] * CELL_SIZE,
            CELL_SIZE, CELL_SIZE
        )
        pygame.draw.rect(screen, GOAL_COLOR, goal_rect)
        
        # Draw hazards
        for hazard in self.maze.hazards:
            hazard_rect = pygame.Rect(
                self.offset_x + hazard[0] * CELL_SIZE,
                self.offset_y + hazard[1] * CELL_SIZE,
                CELL_SIZE, CELL_SIZE
            )
            pygame.draw.rect(screen, HAZARD_COLOR, hazard_rect, border_radius=8)
    
    def draw_panel(self):
        # Draw panel background
        panel_rect = pygame.Rect(SCREEN_WIDTH - PANEL_WIDTH, 0, PANEL_WIDTH, SCREEN_HEIGHT)
        pygame.draw.rect(screen, PANEL_BG, panel_rect)
        pygame.draw.line(screen, (60, 60, 90), (SCREEN_WIDTH - PANEL_WIDTH, 0), 
                         (SCREEN_WIDTH - PANEL_WIDTH, SCREEN_HEIGHT), 3)
        
        # Draw title
        title = font_large.render("RL MAZE", True, TEXT_COLOR)
        screen.blit(title, (SCREEN_WIDTH - PANEL_WIDTH + 20, 20))
        
        # Draw game info
        info_y = 80
        info = [
            f"Lives: {self.player.lives}",
            f"Episodes: {self.episode_count}",
            f"Steps: {self.steps}",
            f"Total Reward: {self.total_reward:.1f}",
            f"Mode: {'Manual' if self.manual_control else 'Auto Training'}",
            f"Time: {self.get_elapsed_time()}s"
        ]
        
        for text in info:
            text_surf = font_medium.render(text, True, TEXT_COLOR)
            screen.blit(text_surf, (SCREEN_WIDTH - PANEL_WIDTH + 20, info_y))
            info_y += 40
        
        # Draw Q-learning parameters
        params_y = info_y + 20
        params = [
            "Q-Learning Parameters:",
            f"Learning Rate: {self.agent.learning_rate}",
            f"Discount Factor: {self.agent.discount_factor}",
            f"Exploration Rate: {self.agent.exploration_rate}"
        ]
        
        for text in params:
            text_surf = font_small.render(text, True, TEXT_COLOR)
            screen.blit(text_surf, (SCREEN_WIDTH - PANEL_WIDTH + 20, params_y))
            params_y += 30
        
        # Draw convergence graph
        if self.convergence_data:
            graph_title = font_medium.render("Convergence Progress", True, TEXT_COLOR)
            screen.blit(graph_title, (SCREEN_WIDTH - PANEL_WIDTH + 20, params_y))
            params_y += 40
            
            # Draw graph background
            graph_rect = pygame.Rect(SCREEN_WIDTH - PANEL_WIDTH + 20, params_y, PANEL_WIDTH - 60, 150)
            pygame.draw.rect(screen, (40, 40, 60), graph_rect)
            pygame.draw.rect(screen, (80, 80, 120), graph_rect, 1)
            
            # Draw data points
            if len(self.convergence_data) > 1:
                max_steps = max(self.convergence_data)
                min_steps = min(self.convergence_data)
                range_steps = max_steps - min_steps if max_steps > min_steps else 1
                
                x_step = (graph_rect.width - 20) / (len(self.convergence_data) - 1)
                points = []
                
                for i, steps in enumerate(self.convergence_data):
                    x = graph_rect.x + 10 + i * x_step
                    y = graph_rect.y + 10 + (steps - min_steps) * 130 / range_steps
                    points.append((x, y))
                
                if len(points) > 1:
                    pygame.draw.lines(screen, (70, 180, 220), False, points, 2)
                
                # Draw labels
                min_text = font_small.render(f"{min_steps}", True, TEXT_COLOR)
                max_text = font_small.render(f"{max_steps}", True, TEXT_COLOR)
                screen.blit(min_text, (graph_rect.right - 30, graph_rect.bottom - 15))
                screen.blit(max_text, (graph_rect.right - 30, graph_rect.top + 5))
        
        # Draw buttons
        for button in self.buttons:
            button.draw(screen)
    
    def draw_message(self, text, color):
        overlay = pygame.Surface((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.SRCALPHA)
        overlay.fill((0, 0, 0, 180))
        screen.blit(overlay, (0, 0))
        
        text_surf = font_large.render(text, True, color)
        text_rect = text_surf.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 - 50))
        screen.blit(text_surf, text_rect)
        
        if self.game_state == "DIED":
            hint = font_medium.render("Press 'Reset Game' to play again", True, TEXT_COLOR)
        else:
            hint = font_medium.render("Press any key to continue", True, TEXT_COLOR)
        
        hint_rect = hint.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 20))
        screen.blit(hint, hint_rect)
    
    def run(self):
        self.running = True
        while self.running:
            if not self.handle_events():
                self.running = False
                break
            self.update()
            self.draw()
            clock.tick(FPS)
        
        pygame.quit()

# Start the game
if __name__ == "__main__":
    game = Game()
    game.run()

#Epochs

In [11]:
import pygame
import random
import numpy as np
import sys
from collections import deque
import math
import atexit
import time

# Initialize pygame
pygame.init()

# Constants
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
CELL_SIZE = 40
GRID_WIDTH = 20
GRID_HEIGHT = 15
PANEL_WIDTH = 400
FPS = 60

# Colors
BACKGROUND = (30, 30, 50)
GRID_LINES = (60, 60, 80)
WALL_COLOR = (50, 50, 70)
PATH_COLOR = (35, 35, 55)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
PANEL_BG = (25, 25, 40)
BUTTON_COLOR = (70, 130, 200)
BUTTON_HOVER = (90, 150, 220)
Q_VALUE_COLORS = [
    (180, 60, 60),    # Low values (red)
    (220, 150, 60),   # Medium values (orange)
    (60, 180, 80)     # High values (green)
]

# Create screen
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Reinforcement Learning Maze Environment")
clock = pygame.time.Clock()

# Fonts
font_large = pygame.font.SysFont("Arial", 36, bold=True)
font_medium = pygame.font.SysFont("Arial", 24)
font_small = pygame.font.SysFont("Arial", 18)

# Register cleanup function
def cleanup():
    pygame.quit()

atexit.register(cleanup)

class Maze:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)  # 0 = path, 1 = wall
        self.start_pos = (1, 1)
        self.goal_pos = (width - 2, height - 2)
        self.hazards = []
        self.generate_maze()
        
    def generate_maze(self):
        # Initialize with walls
        self.grid.fill(1)
        
        # Create a depth-first search maze
        stack = [self.start_pos]
        self.grid[self.start_pos[1]][self.start_pos[0]] = 0
        
        # Possible directions
        directions = [(0, -2), (2, 0), (0, 2), (-2, 0)]  # Up, right, down, left
        
        while stack:
            current = stack[-1]
            x, y = current
            
            # Find unvisited neighbors
            neighbors = []
            for dx, dy in directions:
                nx, ny = x + dx, y + dy
                if 0 < nx < self.width - 1 and 0 < ny < self.height - 1 and self.grid[ny][nx] == 1:
                    neighbors.append((nx, ny, dx, dy))
            
            if neighbors:
                # Choose a random neighbor
                nx, ny, dx, dy = random.choice(neighbors)
                
                # Remove wall between current and neighbor
                self.grid[y + dy//2][x + dx//2] = 0
                self.grid[ny][nx] = 0
                
                # Add to stack
                stack.append((nx, ny))
            else:
                # Backtrack
                stack.pop()
        
        # Ensure there's a path to the goal
        self.create_path_to_goal()
        
        # Add hazards (10% of path cells)
        path_cells = [(x, y) for y in range(self.height) for x in range(self.width) 
                      if self.grid[y][x] == 0 and (x, y) != self.start_pos and (x, y) != self.goal_pos]
        
        num_hazards = max(1, int(len(path_cells) * 0.1))
        self.hazards = random.sample(path_cells, num_hazards)
    
    def create_path_to_goal(self):
        # Use BFS to find a path from start to goal
        queue = deque([self.start_pos])
        visited = {self.start_pos: None}
        
        while queue:
            x, y = queue.popleft()
            
            if (x, y) == self.goal_pos:
                break
                
            for dx, dy in [(0, -1), (1, 0), (0, 1), (-1, 0)]:
                nx, ny = x + dx, y + dy
                if (0 <= nx < self.width and 0 <= ny < self.height and 
                    self.grid[ny][nx] == 0 and (nx, ny) not in visited):
                    queue.append((nx, ny))
                    visited[(nx, ny)] = (x, y)
        
        # If no path found, create one
        if self.goal_pos not in visited:
            x, y = self.goal_pos
            path = []
            
            # Create a path backwards from goal to start
            while (x, y) != self.start_pos:
                path.append((x, y))
                # Move toward start (simplified)
                if x > self.start_pos[0]:
                    x -= 1
                elif x < self.start_pos[0]:
                    x += 1
                elif y > self.start_pos[1]:
                    y -= 1
                elif y < self.start_pos[1]:
                    y += 1
                
                # Remove wall at this position
                if 0 <= x < self.width and 0 <= y < self.height:
                    self.grid[y][x] = 0
            
            # Add the path to the maze
            for pos in path:
                self.grid[pos[1]][pos[0]] = 0
    
    def is_wall(self, x, y):
        return self.grid[y][x] == 1
    
    def is_hazard(self, x, y):
        return (x, y) in self.hazards

class Player:
    def __init__(self, maze):
        self.maze = maze
        self.reset()
        self.lives = 30
        self.speed = 0.2
        self.radius = CELL_SIZE // 3
        self.trail = []
        
    def reset(self):
        self.x, self.y = self.maze.start_pos
        self.px, self.py = self.x, self.y
        self.target_x, self.target_y = self.x, self.y
        self.moving = False
        self.trail = []
        
    def move(self, dx, dy):
        if self.moving:
            return
            
        new_x, new_y = self.x + dx, self.y + dy
        
        # Check boundaries
        if (0 <= new_x < self.maze.width and 0 <= new_y < self.maze.height and 
            not self.maze.is_wall(new_x, new_y)):
            self.px, self.py = self.x, self.y
            self.target_x, self.target_y = new_x, new_y
            self.moving = True
            
            # Add to trail with current timestamp
            self.trail.append((self.x, self.y, time.time()))
    
    def update(self):
        if self.moving:
            # Move toward target
            dx = self.target_x - self.x
            dy = self.target_y - self.y
            dist = math.sqrt(dx*dx + dy*dy)
            
            if dist < self.speed:
                self.x, self.y = self.target_x, self.target_y
                self.moving = False
            else:
                self.x += dx * self.speed
                self.y += dy * self.speed
        
        # Remove trail points older than 1 second
        current_time = time.time()
        self.trail = [(x, y, t) for (x, y, t) in self.trail if current_time - t < 1.0]
    
    def get_position(self):
        return (int(self.x), int(self.y))
    
    def lose_life(self):
        self.lives -= 1
        self.reset()
        return self.lives <= 0
    
    def draw(self, screen, offset_x, offset_y):
        # Draw trail
        current_time = time.time()
        for i, (tx, ty, t) in enumerate(self.trail):
            # Calculate alpha based on remaining time (fade out over 1 second)
            alpha = int(255 * (1 - (current_time - t)))
            if alpha <= 0:
                continue
                
            # Create a surface with per-pixel alpha
            trail_surface = pygame.Surface((CELL_SIZE, CELL_SIZE), pygame.SRCALPHA)
            radius = int(self.radius * (1 - (current_time - t)))
            pygame.draw.circle(trail_surface, (70, 130, 180, alpha), 
                              (CELL_SIZE // 2, CELL_SIZE // 2), 
                              radius)
            screen.blit(trail_surface, (offset_x + tx * CELL_SIZE, offset_y + ty * CELL_SIZE))
        
        # Draw player
        pygame.draw.circle(screen, PLAYER_COLOR, 
                          (offset_x + (self.x + 0.5) * CELL_SIZE, offset_y + (self.y + 0.5) * CELL_SIZE), 
                          self.radius)
        
        # Draw player direction indicator
        if self.moving:
            dx = self.target_x - self.px
            dy = self.target_y - self.py
            if dx != 0 or dy != 0:
                angle = math.atan2(dy, dx)
                end_x = (self.x + 0.5 + 0.6 * math.cos(angle)) * CELL_SIZE
                end_y = (self.y + 0.5 + 0.6 * math.sin(angle)) * CELL_SIZE
                pygame.draw.line(screen, (255, 255, 255), 
                                (offset_x + (self.x + 0.5) * CELL_SIZE, offset_y + (self.y + 0.5) * CELL_SIZE),
                                (offset_x + end_x, offset_y + end_y), 3)

class QLearningAgent:
    def __init__(self, maze, player):
        self.maze = maze
        self.player = player
        self.q_table = {}
        self.learning_rate = 0.1
        self.discount_factor = 0.95
        self.exploration_rate = 0.3
        self.state = None
        self.action = None
        
    def get_state(self):
        return self.player.get_position()
    
    def get_q_value(self, state, action):
        if state not in self.q_table:
            self.q_table[state] = [0.0, 0.0, 0.0, 0.0]  # Up, Right, Down, Left
        return self.q_table[state][action]
    
    def choose_action(self, state):
        # Exploration vs exploitation
        if random.random() < self.exploration_rate:
            return random.randint(0, 3)  # Random action
        else:
            # Choose best action
            q_values = [self.get_q_value(state, a) for a in range(4)]
            max_q = max(q_values)
            # If multiple actions have the same max q value, choose randomly among them
            actions = [a for a, q in enumerate(q_values) if q == max_q]
            return random.choice(actions)
    
    def update_q_value(self, state, action, reward, next_state):
        current_q = self.get_q_value(state, action)
        next_max_q = max([self.get_q_value(next_state, a) for a in range(4)])
        
        # Q-learning formula
        new_q = current_q + self.learning_rate * (reward + self.discount_factor * next_max_q - current_q)
        self.q_table[state][action] = new_q
    
    def start_episode(self):
        self.state = self.get_state()
        self.action = self.choose_action(self.state)
        return self.action
    
    def step(self, reward, next_state):
        # Update Q-value for previous state-action pair
        self.update_q_value(self.state, self.action, reward, next_state)
        
        # Choose next action
        self.state = next_state
        self.action = self.choose_action(self.state)
        return self.action
    
    def get_q_values(self):
        return self.q_table

class Button:
    def __init__(self, x, y, width, height, text, action=None):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.action = action
        self.hovered = False
        
    def draw(self, screen):
        color = BUTTON_HOVER if self.hovered else BUTTON_COLOR
        pygame.draw.rect(screen, color, self.rect, border_radius=8)
        pygame.draw.rect(screen, (100, 160, 230), self.rect, 2, border_radius=8)
        
        text_surf = font_medium.render(self.text, True, TEXT_COLOR)
        text_rect = text_surf.get_rect(center=self.rect.center)
        screen.blit(text_surf, text_rect)
        
    def check_hover(self, pos):
        self.hovered = self.rect.collidepoint(pos)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered and self.action:
                return self.action()
        return None

class Game:
    def __init__(self):
        self.maze = Maze(GRID_WIDTH, GRID_HEIGHT)
        self.player = Player(self.maze)
        self.agent = QLearningAgent(self.maze, self.player)
        self.game_state = "PLAYING"  # PLAYING, DIED, WON
        self.offset_x = 50
        self.offset_y = 50
        self.manual_control = True
        self.test_mode = False  # New flag for test mode
        self.episode_count = 0
        self.steps = 0
        self.episode_reward = 0
        self.total_reward = 0
        self.convergence_data = []
        self.running = True
        self.start_time = time.time()
        
        # Create buttons
        button_width = 150
        button_height = 40
        button_spacing = 20
        panel_center = SCREEN_WIDTH - PANEL_WIDTH // 2
        self.buttons = [
            Button(panel_center - button_width//2, 500, button_width, button_height, "Manual Control", self.set_manual),
            Button(panel_center - button_width//2, 500 + button_height + button_spacing, button_width, button_height, "Auto Training", self.set_auto),
            Button(panel_center - button_width//2, 500 + 2*(button_height + button_spacing), button_width, button_height, "Test Mode", self.set_test),
            Button(panel_center - button_width//2, 500 + 3*(button_height + button_spacing), button_width, button_height, "Reset Game", self.reset_game),
            Button(panel_center - button_width//2, 500 + 4*(button_height + button_spacing), button_width, button_height, "Quit", self.quit_game)
        ]
    
    def set_manual(self):
        self.manual_control = True
        self.test_mode = False
        return True
    
    def set_auto(self):
        self.manual_control = False
        self.test_mode = False
        return True
    
    def set_test(self):
        self.manual_control = False
        self.test_mode = True
        return True
    
    def reset_game(self):
        self.maze = Maze(GRID_WIDTH, GRID_HEIGHT)
        self.player = Player(self.maze)
        self.agent = QLearningAgent(self.maze, self.player)
        self.game_state = "PLAYING"
        self.episode_count = 0
        self.steps = 0
        self.episode_reward = 0
        self.total_reward = 0
        self.convergence_data = []
        self.start_time = time.time()
        return True
    
    def quit_game(self):
        self.running = False
        return True
    
    def get_elapsed_time(self):
        return int(time.time() - self.start_time)
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.running = False
                return False
                
            # Handle button events
            for button in self.buttons:
                button.handle_event(event)
            
            # Handle keyboard events
            if event.type == pygame.KEYDOWN and self.game_state == "PLAYING":
                if event.key == pygame.K_UP and self.manual_control:
                    self.player.move(0, -1)
                elif event.key == pygame.K_RIGHT and self.manual_control:
                    self.player.move(1, 0)
                elif event.key == pygame.K_DOWN and self.manual_control:
                    self.player.move(0, 1)
                elif event.key == pygame.K_LEFT and self.manual_control:
                    self.player.move(-1, 0)
                elif event.key == pygame.K_r:
                    self.player.reset()
                elif event.key == pygame.K_SPACE:
                    self.manual_control = not self.manual_control
                elif event.key == pygame.K_ESCAPE:
                    self.running = False
                    return False
                    
            # Handle mouse events for buttons
            if event.type == pygame.MOUSEMOTION:
                for button in self.buttons:
                    button.check_hover(event.pos)
        
        return True
    
    def update(self):
        if self.game_state != "PLAYING":
            # Automatically continue to next episode
            if self.game_state == "WON" or (self.game_state == "DIED" and not self.test_mode):
                self.player.reset()
                self.game_state = "PLAYING"
            return
            
        self.player.update()
        player_pos = self.player.get_position()
        
        # Check for hazard collision
        if self.maze.is_hazard(player_pos[0], player_pos[1]):
            # Only lose lives in test mode or manual control
            if self.test_mode or self.manual_control:
                dead = self.player.lose_life()
                if dead:
                    self.game_state = "DIED"
            self.episode_reward -= 50
        
        # Check for goal reached
        if player_pos == self.maze.goal_pos:
            self.episode_count += 1
            self.episode_reward += 100
            self.convergence_data.append(self.steps)
            self.steps = 0
            self.game_state = "WON"
            
            # In auto training mode, reset immediately
            if not self.test_mode:
                self.player.reset()
                self.game_state = "PLAYING"
        
        # Auto control (Q-learning)
        if not self.manual_control and not self.player.moving:
            # Get current state
            current_state = self.agent.get_state()
            
            # Choose and perform action
            if self.agent.state is None:
                action = self.agent.start_episode()
            else:
                # Calculate reward
                reward = -0.1  # Small penalty for each step
                self.episode_reward += reward
                self.total_reward += reward
                
                # Update Q-value and get next action
                action = self.agent.step(reward, current_state)
            
            # Perform action
            if action == 0:  # Up
                self.player.move(0, -1)
            elif action == 1:  # Right
                self.player.move(1, 0)
            elif action == 2:  # Down
                self.player.move(0, 1)
            elif action == 3:  # Left
                self.player.move(-1, 0)
            
            self.steps += 1
    
    def draw(self):
        screen.fill(BACKGROUND)
        
        # Draw maze and player
        self.draw_maze()
        self.player.draw(screen, self.offset_x, self.offset_y)
        
        # Draw right panel
        self.draw_panel()
        
        # Draw game state messages
        if self.game_state == "DIED":
            self.draw_message("YOU DIED!", HAZARD_COLOR)
        elif self.game_state == "WON":
            self.draw_message("LEVEL COMPLETE!", GOAL_COLOR)
        
        pygame.display.flip()
    
    def draw_maze(self):
        # Draw maze grid
        for y in range(self.maze.height):
            for x in range(self.maze.width):
                rect = pygame.Rect(
                    self.offset_x + x * CELL_SIZE,
                    self.offset_y + y * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                if self.maze.is_wall(x, y):
                    pygame.draw.rect(screen, WALL_COLOR, rect)
                else:
                    pygame.draw.rect(screen, PATH_COLOR, rect)
                
                # Draw grid lines
                pygame.draw.rect(screen, GRID_LINES, rect, 1)
        
        # Draw start and goal
        start_rect = pygame.Rect(
            self.offset_x + self.maze.start_pos[0] * CELL_SIZE,
            self.offset_y + self.maze.start_pos[1] * CELL_SIZE,
            CELL_SIZE, CELL_SIZE
        )
        pygame.draw.rect(screen, (70, 100, 150), start_rect)
        
        goal_rect = pygame.Rect(
            self.offset_x + self.maze.goal_pos[0] * CELL_SIZE,
            self.offset_y + self.maze.goal_pos[1] * CELL_SIZE,
            CELL_SIZE, CELL_SIZE
        )
        pygame.draw.rect(screen, GOAL_COLOR, goal_rect)
        
        # Draw hazards
        for hazard in self.maze.hazards:
            hazard_rect = pygame.Rect(
                self.offset_x + hazard[0] * CELL_SIZE,
                self.offset_y + hazard[1] * CELL_SIZE,
                CELL_SIZE, CELL_SIZE
            )
            pygame.draw.rect(screen, HAZARD_COLOR, hazard_rect, border_radius=8)
    
    def draw_panel(self):
        # Draw panel background
        panel_rect = pygame.Rect(SCREEN_WIDTH - PANEL_WIDTH, 0, PANEL_WIDTH, SCREEN_HEIGHT)
        pygame.draw.rect(screen, PANEL_BG, panel_rect)
        pygame.draw.line(screen, (60, 60, 90), (SCREEN_WIDTH - PANEL_WIDTH, 0), 
                         (SCREEN_WIDTH - PANEL_WIDTH, SCREEN_HEIGHT), 3)
        
        # Draw title
        title = font_large.render("RL MAZE", True, TEXT_COLOR)
        screen.blit(title, (SCREEN_WIDTH - PANEL_WIDTH + 20, 20))
        
        # Draw game info
        info_y = 80
        info = [
            f"Lives: {self.player.lives}",
            f"Episodes: {self.episode_count}",
            f"Steps: {self.steps}",
            f"Total Reward: {self.total_reward:.1f}",
            f"Mode: {'Manual' if self.manual_control else 'Test' if self.test_mode else 'Auto Training'}",
            f"Time: {self.get_elapsed_time()}s"
        ]
        
        for text in info:
            text_surf = font_medium.render(text, True, TEXT_COLOR)
            screen.blit(text_surf, (SCREEN_WIDTH - PANEL_WIDTH + 20, info_y))
            info_y += 40
        
        # Draw Q-learning parameters
        params_y = info_y + 20
        params = [
            "Q-Learning Parameters:",
            f"Learning Rate: {self.agent.learning_rate}",
            f"Discount Factor: {self.agent.discount_factor}",
            f"Exploration Rate: {self.agent.exploration_rate}"
        ]
        
        for text in params:
            text_surf = font_small.render(text, True, TEXT_COLOR)
            screen.blit(text_surf, (SCREEN_WIDTH - PANEL_WIDTH + 20, params_y))
            params_y += 30
        
        # Draw convergence graph
        if self.convergence_data:
            graph_title = font_medium.render("Convergence Progress", True, TEXT_COLOR)
            screen.blit(graph_title, (SCREEN_WIDTH - PANEL_WIDTH + 20, params_y))
            params_y += 40
            
            # Draw graph background
            graph_rect = pygame.Rect(SCREEN_WIDTH - PANEL_WIDTH + 20, params_y, PANEL_WIDTH - 60, 150)
            pygame.draw.rect(screen, (40, 40, 60), graph_rect)
            pygame.draw.rect(screen, (80, 80, 120), graph_rect, 1)
            
            # Draw data points
            if len(self.convergence_data) > 1:
                max_steps = max(self.convergence_data)
                min_steps = min(self.convergence_data)
                range_steps = max_steps - min_steps if max_steps > min_steps else 1
                
                x_step = (graph_rect.width - 20) / (len(self.convergence_data) - 1)
                points = []
                
                for i, steps in enumerate(self.convergence_data):
                    x = graph_rect.x + 10 + i * x_step
                    y = graph_rect.y + 10 + (steps - min_steps) * 130 / range_steps
                    points.append((x, y))
                
                if len(points) > 1:
                    pygame.draw.lines(screen, (70, 180, 220), False, points, 2)
                
                # Draw labels
                min_text = font_small.render(f"{min_steps}", True, TEXT_COLOR)
                max_text = font_small.render(f"{max_steps}", True, TEXT_COLOR)
                screen.blit(min_text, (graph_rect.right - 30, graph_rect.bottom - 15))
                screen.blit(max_text, (graph_rect.right - 30, graph_rect.top + 5))
        
        # Draw buttons
        for button in self.buttons:
            button.draw(screen)
    
    def draw_message(self, text, color):
        overlay = pygame.Surface((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.SRCALPHA)
        overlay.fill((0, 0, 0, 180))
        screen.blit(overlay, (0, 0))
        
        text_surf = font_large.render(text, True, color)
        text_rect = text_surf.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 - 50))
        screen.blit(text_surf, text_rect)
        
        if self.game_state == "DIED":
            hint = font_medium.render("Press 'Reset Game' to play again", True, TEXT_COLOR)
        else:
            hint = font_medium.render("Press any key to continue", True, TEXT_COLOR)
        
        hint_rect = hint.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 20))
        screen.blit(hint, hint_rect)
    
    def run(self):
        self.running = True
        while self.running:
            if not self.handle_events():
                self.running = False
                break
            self.update()
            self.draw()
            clock.tick(FPS)
        
        pygame.quit()

# Start the game
if __name__ == "__main__":
    game = Game()
    game.run()

#speed learning

In [13]:
import pygame
import random
import numpy as np
import sys
from collections import deque
import math
import atexit
import time

# Initialize pygame
pygame.init()

# Constants
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
CELL_SIZE = 40
GRID_WIDTH = 20
GRID_HEIGHT = 15
PANEL_WIDTH = 400
FPS = 60

# Colors
BACKGROUND = (30, 30, 50)
GRID_LINES = (60, 60, 80)
WALL_COLOR = (50, 50, 70)
PATH_COLOR = (35, 35, 55)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
PANEL_BG = (25, 25, 40)
BUTTON_COLOR = (70, 130, 200)
BUTTON_HOVER = (90, 150, 220)
Q_VALUE_COLORS = [
    (180, 60, 60),    # Low values (red)
    (220, 150, 60),   # Medium values (orange)
    (60, 180, 80)     # High values (green)
]

# Create screen
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Reinforcement Learning Maze Environment")
clock = pygame.time.Clock()

# Fonts
font_large = pygame.font.SysFont("Arial", 36, bold=True)
font_medium = pygame.font.SysFont("Arial", 24)
font_small = pygame.font.SysFont("Arial", 18)

# Register cleanup function
def cleanup():
    pygame.quit()

atexit.register(cleanup)

class Maze:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)  # 0 = path, 1 = wall
        self.start_pos = (1, 1)
        self.goal_pos = (width - 2, height - 2)
        self.hazards = []
        self.generate_maze()
        
    def generate_maze(self):
        # Initialize with walls
        self.grid.fill(1)
        
        # Create a depth-first search maze
        stack = [self.start_pos]
        self.grid[self.start_pos[1]][self.start_pos[0]] = 0
        
        # Possible directions
        directions = [(0, -2), (2, 0), (0, 2), (-2, 0)]  # Up, right, down, left
        
        while stack:
            current = stack[-1]
            x, y = current
            
            # Find unvisited neighbors
            neighbors = []
            for dx, dy in directions:
                nx, ny = x + dx, y + dy
                if 0 < nx < self.width - 1 and 0 < ny < self.height - 1 and self.grid[ny][nx] == 1:
                    neighbors.append((nx, ny, dx, dy))
            
            if neighbors:
                # Choose a random neighbor
                nx, ny, dx, dy = random.choice(neighbors)
                
                # Remove wall between current and neighbor
                self.grid[y + dy//2][x + dx//2] = 0
                self.grid[ny][nx] = 0
                
                # Add to stack
                stack.append((nx, ny))
            else:
                # Backtrack
                stack.pop()
        
        # Ensure there's a path to the goal
        self.create_path_to_goal()
        
        # Add hazards (10% of path cells)
        path_cells = [(x, y) for y in range(self.height) for x in range(self.width) 
                      if self.grid[y][x] == 0 and (x, y) != self.start_pos and (x, y) != self.goal_pos]
        
        num_hazards = max(1, int(len(path_cells) * 0.1))
        self.hazards = random.sample(path_cells, num_hazards)
    
    def create_path_to_goal(self):
        # Use BFS to find a path from start to goal
        queue = deque([self.start_pos])
        visited = {self.start_pos: None}
        
        while queue:
            x, y = queue.popleft()
            
            if (x, y) == self.goal_pos:
                break
                
            for dx, dy in [(0, -1), (1, 0), (0, 1), (-1, 0)]:
                nx, ny = x + dx, y + dy
                if (0 <= nx < self.width and 0 <= ny < self.height and 
                    self.grid[ny][nx] == 0 and (nx, ny) not in visited):
                    queue.append((nx, ny))
                    visited[(nx, ny)] = (x, y)
        
        # If no path found, create one
        if self.goal_pos not in visited:
            x, y = self.goal_pos
            path = []
            
            # Create a path backwards from goal to start
            while (x, y) != self.start_pos:
                path.append((x, y))
                # Move toward start (simplified)
                if x > self.start_pos[0]:
                    x -= 1
                elif x < self.start_pos[0]:
                    x += 1
                elif y > self.start_pos[1]:
                    y -= 1
                elif y < self.start_pos[1]:
                    y += 1
                
                # Remove wall at this position
                if 0 <= x < self.width and 0 <= y < self.height:
                    self.grid[y][x] = 0
            
            # Add the path to the maze
            for pos in path:
                self.grid[pos[1]][pos[0]] = 0
    
    def is_wall(self, x, y):
        return self.grid[y][x] == 1
    
    def is_hazard(self, x, y):
        return (x, y) in self.hazards

class Player:
    def __init__(self, maze):
        self.maze = maze
        self.reset()
        self.lives = 30
        self.speed = 0.2
        self.radius = CELL_SIZE // 3
        self.trail = []
        
    def reset(self):
        self.x, self.y = self.maze.start_pos
        self.px, self.py = self.x, self.y
        self.target_x, self.target_y = self.x, self.y
        self.moving = False
        self.trail = []
        
    def move(self, dx, dy):
        if self.moving:
            return
            
        new_x, new_y = self.x + dx, self.y + dy
        
        # Check boundaries
        if (0 <= new_x < self.maze.width and 0 <= new_y < self.maze.height and 
            not self.maze.is_wall(new_x, new_y)):
            self.px, self.py = self.x, self.y
            self.target_x, self.target_y = new_x, new_y
            self.moving = True
            
            # Add to trail with current timestamp
            self.trail.append((self.x, self.y, time.time()))
    
    def update(self):
        if self.moving:
            # Move toward target
            dx = self.target_x - self.x
            dy = self.target_y - self.y
            dist = math.sqrt(dx*dx + dy*dy)
            
            if dist < self.speed:
                self.x, self.y = self.target_x, self.target_y
                self.moving = False
            else:
                self.x += dx * self.speed
                self.y += dy * self.speed
        
        # Remove trail points older than 1 second
        current_time = time.time()
        self.trail = [(x, y, t) for (x, y, t) in self.trail if current_time - t < 1.0]
    
    def get_position(self):
        return (int(self.x), int(self.y))
    
    def lose_life(self):
        self.lives -= 1
        self.reset()
        return self.lives <= 0
    
    def draw(self, screen, offset_x, offset_y):
        # Draw trail
        current_time = time.time()
        for i, (tx, ty, t) in enumerate(self.trail):
            # Calculate alpha based on remaining time (fade out over 1 second)
            alpha = int(255 * (1 - (current_time - t)))
            if alpha <= 0:
                continue
                
            # Create a surface with per-pixel alpha
            trail_surface = pygame.Surface((CELL_SIZE, CELL_SIZE), pygame.SRCALPHA)
            radius = int(self.radius * (1 - (current_time - t)))
            pygame.draw.circle(trail_surface, (70, 130, 180, alpha), 
                              (CELL_SIZE // 2, CELL_SIZE // 2), 
                              radius)
            screen.blit(trail_surface, (offset_x + tx * CELL_SIZE, offset_y + ty * CELL_SIZE))
        
        # Draw player
        pygame.draw.circle(screen, PLAYER_COLOR, 
                          (offset_x + (self.x + 0.5) * CELL_SIZE, offset_y + (self.y + 0.5) * CELL_SIZE), 
                          self.radius)
        
        # Draw player direction indicator
        if self.moving:
            dx = self.target_x - self.px
            dy = self.target_y - self.py
            if dx != 0 or dy != 0:
                angle = math.atan2(dy, dx)
                end_x = (self.x + 0.5 + 0.6 * math.cos(angle)) * CELL_SIZE
                end_y = (self.y + 0.5 + 0.6 * math.sin(angle)) * CELL_SIZE
                pygame.draw.line(screen, (255, 255, 255), 
                                (offset_x + (self.x + 0.5) * CELL_SIZE, offset_y + (self.y + 0.5) * CELL_SIZE),
                                (offset_x + end_x, offset_y + end_y), 3)

class QLearningAgent:
    def __init__(self, maze, player):
        self.maze = maze
        self.player = player
        self.q_table = {}
        self.learning_rate = 0.1
        self.discount_factor = 0.95
        self.exploration_rate = 0.3
        self.state = None
        self.action = None
        
    def get_state(self):
        return self.player.get_position()
    
    def get_q_value(self, state, action):
        if state not in self.q_table:
            self.q_table[state] = [0.0, 0.0, 0.0, 0.0]  # Up, Right, Down, Left
        return self.q_table[state][action]
    
    def choose_action(self, state):
        # Exploration vs exploitation
        if random.random() < self.exploration_rate:
            return random.randint(0, 3)  # Random action
        else:
            # Choose best action
            q_values = [self.get_q_value(state, a) for a in range(4)]
            max_q = max(q_values)
            # If multiple actions have the same max q value, choose randomly among them
            actions = [a for a, q in enumerate(q_values) if q == max_q]
            return random.choice(actions)
    
    def update_q_value(self, state, action, reward, next_state):
        current_q = self.get_q_value(state, action)
        next_max_q = max([self.get_q_value(next_state, a) for a in range(4)])
        
        # Q-learning formula
        new_q = current_q + self.learning_rate * (reward + self.discount_factor * next_max_q - current_q)
        self.q_table[state][action] = new_q
    
    def start_episode(self):
        self.state = self.get_state()
        self.action = self.choose_action(self.state)
        return self.action
    
    def step(self, reward, next_state):
        # Update Q-value for previous state-action pair
        self.update_q_value(self.state, self.action, reward, next_state)
        
        # Choose next action
        self.state = next_state
        self.action = self.choose_action(self.state)
        return self.action
    
    def get_q_values(self):
        return self.q_table

class Button:
    def __init__(self, x, y, width, height, text, action=None):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.action = action
        self.hovered = False
        
    def draw(self, screen):
        color = BUTTON_HOVER if self.hovered else BUTTON_COLOR
        pygame.draw.rect(screen, color, self.rect, border_radius=8)
        pygame.draw.rect(screen, (100, 160, 230), self.rect, 2, border_radius=8)
        
        text_surf = font_medium.render(self.text, True, TEXT_COLOR)
        text_rect = text_surf.get_rect(center=self.rect.center)
        screen.blit(text_surf, text_rect)
        
    def check_hover(self, pos):
        self.hovered = self.rect.collidepoint(pos)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered and self.action:
                return self.action()
        return None

class Game:
    def __init__(self):
        self.maze = Maze(GRID_WIDTH, GRID_HEIGHT)
        self.player = Player(self.maze)
        self.agent = QLearningAgent(self.maze, self.player)
        self.game_state = "PLAYING"  # PLAYING, DIED, WON
        self.offset_x = 50
        self.offset_y = 50
        self.manual_control = True
        self.test_mode = False  # New flag for test mode
        self.episode_count = 0
        self.steps = 0
        self.episode_reward = 0
        self.total_reward = 0
        self.convergence_data = []
        self.running = True
        self.start_time = time.time()
        self.simulation_speed = 1  # 1x normal speed
        self.speed_multipliers = [1, 2, 5, 10, 20, 50, 100, 500, 1000]  # Available speed multipliers
        self.current_speed_index = 0  # Start with normal speed
        
        # Create buttons
        button_width = 150
        button_height = 40
        button_spacing = 20
        panel_center = SCREEN_WIDTH - PANEL_WIDTH // 2
        self.buttons = [
            Button(panel_center - button_width//2, 500, button_width, button_height, "Manual Control", self.set_manual),
            Button(panel_center - button_width//2, 500 + button_height + button_spacing, button_width, button_height, "Auto Training", self.set_auto),
            Button(panel_center - button_width//2, 500 + 2*(button_height + button_spacing), button_width, button_height, "Test Mode", self.set_test),
            Button(panel_center - button_width//2, 500 + 3*(button_height + button_spacing), button_width, button_height, "Reset Game", self.reset_game),
            Button(panel_center - button_width//2, 500 + 4*(button_height + button_spacing), button_width, button_height, "Quit", self.quit_game),
            Button(panel_center - button_width//2, 500 + 5*(button_height + button_spacing), button_width, button_height, f"Speed: {self.simulation_speed}x", self.toggle_speed)
        ]
    
    def set_manual(self):
        self.manual_control = True
        self.test_mode = False
        return True
    
    def set_auto(self):
        self.manual_control = False
        self.test_mode = False
        return True
    
    def set_test(self):
        self.manual_control = False
        self.test_mode = True
        return True
    
    def reset_game(self):
        self.maze = Maze(GRID_WIDTH, GRID_HEIGHT)
        self.player = Player(self.maze)
        self.agent = QLearningAgent(self.maze, self.player)
        self.game_state = "PLAYING"
        self.episode_count = 0
        self.steps = 0
        self.episode_reward = 0
        self.total_reward = 0
        self.convergence_data = []
        self.start_time = time.time()
        return True
    
    def quit_game(self):
        self.running = False
        return True
    
    def toggle_speed(self):
        self.current_speed_index = (self.current_speed_index + 1) % len(self.speed_multipliers)
        self.simulation_speed = self.speed_multipliers[self.current_speed_index]
        self.buttons[5].text = f"Speed: {self.simulation_speed}x"
        return True
    
    def get_elapsed_time(self):
        return int(time.time() - self.start_time)
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.running = False
                return False
                
            # Handle button events
            for button in self.buttons:
                result = button.handle_event(event)
                if result is not None and button.text.startswith("Speed:"):
                    # Update speed button text
                    button.text = f"Speed: {self.simulation_speed}x"
            
            # Handle keyboard events
            if event.type == pygame.KEYDOWN and self.game_state == "PLAYING":
                if event.key == pygame.K_UP and self.manual_control:
                    self.player.move(0, -1)
                elif event.key == pygame.K_RIGHT and self.manual_control:
                    self.player.move(1, 0)
                elif event.key == pygame.K_DOWN and self.manual_control:
                    self.player.move(0, 1)
                elif event.key == pygame.K_LEFT and self.manual_control:
                    self.player.move(-1, 0)
                elif event.key == pygame.K_r:
                    self.player.reset()
                elif event.key == pygame.K_SPACE:
                    self.manual_control = not self.manual_control
                elif event.key == pygame.K_ESCAPE:
                    self.running = False
                    return False
                elif event.key == pygame.K_PLUS or event.key == pygame.K_EQUALS:
                    self.toggle_speed()
                elif event.key == pygame.K_MINUS:
                    self.current_speed_index = (self.current_speed_index - 1) % len(self.speed_multipliers)
                    self.simulation_speed = self.speed_multipliers[self.current_speed_index]
                    self.buttons[5].text = f"Speed: {self.simulation_speed}x"
                    
            # Handle mouse events for buttons
            if event.type == pygame.MOUSEMOTION:
                for button in self.buttons:
                    button.check_hover(event.pos)
        
        return True
    
    def update(self):
        if self.game_state != "PLAYING":
            # Automatically continue to next episode
            if self.game_state == "WON" or (self.game_state == "DIED" and not self.test_mode):
                self.player.reset()
                self.game_state = "PLAYING"
            return
            
        self.player.update()
        player_pos = self.player.get_position()
        
        # Check for hazard collision
        if self.maze.is_hazard(player_pos[0], player_pos[1]):
            self.episode_reward -= 50
            self.total_reward -= 50
            
            # In auto training mode, reset immediately with penalty
            if not self.test_mode and not self.manual_control:
                self.player.reset()
            # In test mode or manual control, lose a life
            elif self.test_mode or self.manual_control:
                dead = self.player.lose_life()
                if dead:
                    self.game_state = "DIED"
        
        # Check for goal reached
        if player_pos == self.maze.goal_pos:
            self.episode_count += 1
            self.episode_reward += 100
            self.total_reward += 100
            self.convergence_data.append(self.steps)
            self.steps = 0
            self.game_state = "WON"
            
            # In auto training mode, reset immediately
            if not self.test_mode:
                self.player.reset()
                self.game_state = "PLAYING"
        
        # Auto control (Q-learning)
        if not self.manual_control and not self.player.moving:
            # Get current state
            current_state = self.agent.get_state()
            
            # Choose and perform action
            if self.agent.state is None:
                action = self.agent.start_episode()
            else:
                # Calculate reward
                reward = -0.1  # Small penalty for each step
                self.episode_reward += reward
                self.total_reward += reward
                
                # Update Q-value and get next action
                action = self.agent.step(reward, current_state)
            
            # Perform action
            if action == 0:  # Up
                self.player.move(0, -1)
            elif action == 1:  # Right
                self.player.move(1, 0)
            elif action == 2:  # Down
                self.player.move(0, 1)
            elif action == 3:  # Left
                self.player.move(-1, 0)
            
            self.steps += 1
    
    def draw(self):
        screen.fill(BACKGROUND)
        
        # Draw maze and player
        self.draw_maze()
        self.player.draw(screen, self.offset_x, self.offset_y)
        
        # Draw right panel
        self.draw_panel()
        
        # Draw game state messages
        if self.game_state == "DIED":
            self.draw_message("YOU DIED!", HAZARD_COLOR)
        elif self.game_state == "WON":
            self.draw_message("LEVEL COMPLETE!", GOAL_COLOR)
        
        pygame.display.flip()
        # Adjust FPS based on simulation speed
        clock.tick(FPS * self.simulation_speed)
    
    def draw_maze(self):
        # Draw maze grid
        for y in range(self.maze.height):
            for x in range(self.maze.width):
                rect = pygame.Rect(
                    self.offset_x + x * CELL_SIZE,
                    self.offset_y + y * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                if self.maze.is_wall(x, y):
                    pygame.draw.rect(screen, WALL_COLOR, rect)
                else:
                    pygame.draw.rect(screen, PATH_COLOR, rect)
                
                # Draw grid lines
                pygame.draw.rect(screen, GRID_LINES, rect, 1)
        
        # Draw start and goal
        start_rect = pygame.Rect(
            self.offset_x + self.maze.start_pos[0] * CELL_SIZE,
            self.offset_y + self.maze.start_pos[1] * CELL_SIZE,
            CELL_SIZE, CELL_SIZE
        )
        pygame.draw.rect(screen, (70, 100, 150), start_rect)
        
        goal_rect = pygame.Rect(
            self.offset_x + self.maze.goal_pos[0] * CELL_SIZE,
            self.offset_y + self.maze.goal_pos[1] * CELL_SIZE,
            CELL_SIZE, CELL_SIZE
        )
        pygame.draw.rect(screen, GOAL_COLOR, goal_rect)
        
        # Draw hazards
        for hazard in self.maze.hazards:
            hazard_rect = pygame.Rect(
                self.offset_x + hazard[0] * CELL_SIZE,
                self.offset_y + hazard[1] * CELL_SIZE,
                CELL_SIZE, CELL_SIZE
            )
            pygame.draw.rect(screen, HAZARD_COLOR, hazard_rect, border_radius=8)
    
    def draw_panel(self):
        # Draw panel background
        panel_rect = pygame.Rect(SCREEN_WIDTH - PANEL_WIDTH, 0, PANEL_WIDTH, SCREEN_HEIGHT)
        pygame.draw.rect(screen, PANEL_BG, panel_rect)
        pygame.draw.line(screen, (60, 60, 90), (SCREEN_WIDTH - PANEL_WIDTH, 0), 
                         (SCREEN_WIDTH - PANEL_WIDTH, SCREEN_HEIGHT), 3)
        
        # Draw title
        title = font_large.render("RL MAZE", True, TEXT_COLOR)
        screen.blit(title, (SCREEN_WIDTH - PANEL_WIDTH + 20, 20))
        
        # Draw game info
        info_y = 80
        info = [
            f"Lives: {self.player.lives}",
            f"Episodes: {self.episode_count}",
            f"Steps: {self.steps}",
            f"Total Reward: {self.total_reward:.1f}",
            f"Mode: {'Manual' if self.manual_control else 'Test' if self.test_mode else 'Auto Training'}",
            f"Time: {self.get_elapsed_time()}s",
            f"Speed: {self.simulation_speed}x"
        ]
        
        for text in info:
            text_surf = font_medium.render(text, True, TEXT_COLOR)
            screen.blit(text_surf, (SCREEN_WIDTH - PANEL_WIDTH + 20, info_y))
            info_y += 40
        
        # Draw Q-learning parameters
        params_y = info_y + 20
        params = [
            "Q-Learning Parameters:",
            f"Learning Rate: {self.agent.learning_rate}",
            f"Discount Factor: {self.agent.discount_factor}",
            f"Exploration Rate: {self.agent.exploration_rate}"
        ]
        
        for text in params:
            text_surf = font_small.render(text, True, TEXT_COLOR)
            screen.blit(text_surf, (SCREEN_WIDTH - PANEL_WIDTH + 20, params_y))
            params_y += 30
        
        # Draw convergence graph
        if self.convergence_data:
            graph_title = font_medium.render("Convergence Progress", True, TEXT_COLOR)
            screen.blit(graph_title, (SCREEN_WIDTH - PANEL_WIDTH + 20, params_y))
            params_y += 40
            
            # Draw graph background
            graph_rect = pygame.Rect(SCREEN_WIDTH - PANEL_WIDTH + 20, params_y, PANEL_WIDTH - 60, 150)
            pygame.draw.rect(screen, (40, 40, 60), graph_rect)
            pygame.draw.rect(screen, (80, 80, 120), graph_rect, 1)
            
            # Draw data points
            if len(self.convergence_data) > 1:
                max_steps = max(self.convergence_data)
                min_steps = min(self.convergence_data)
                range_steps = max_steps - min_steps if max_steps > min_steps else 1
                
                x_step = (graph_rect.width - 20) / (len(self.convergence_data) - 1)
                points = []
                
                for i, steps in enumerate(self.convergence_data):
                    x = graph_rect.x + 10 + i * x_step
                    y = graph_rect.y + 10 + (steps - min_steps) * 130 / range_steps
                    points.append((x, y))
                
                if len(points) > 1:
                    pygame.draw.lines(screen, (70, 180, 220), False, points, 2)
                
                # Draw labels
                min_text = font_small.render(f"{min_steps}", True, TEXT_COLOR)
                max_text = font_small.render(f"{max_steps}", True, TEXT_COLOR)
                screen.blit(min_text, (graph_rect.right - 30, graph_rect.bottom - 15))
                screen.blit(max_text, (graph_rect.right - 30, graph_rect.top + 5))
        
        # Draw buttons
        for button in self.buttons:
            button.draw(screen)
    
    def draw_message(self, text, color):
        overlay = pygame.Surface((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.SRCALPHA)
        overlay.fill((0, 0, 0, 180))
        screen.blit(overlay, (0, 0))
        
        text_surf = font_large.render(text, True, color)
        text_rect = text_surf.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 - 50))
        screen.blit(text_surf, text_rect)
        
        if self.game_state == "DIED":
            hint = font_medium.render("Press 'Reset Game' to play again", True, TEXT_COLOR)
        else:
            hint = font_medium.render("Press any key to continue", True, TEXT_COLOR)
        
        hint_rect = hint.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 20))
        screen.blit(hint, hint_rect)
    
    def run(self):
        self.running = True
        while self.running:
            if not self.handle_events():
                self.running = False
                break
            self.update()
            self.draw()
        
        pygame.quit()

# Start the game
if __name__ == "__main__":
    game = Game()
    game.run()

In [15]:
import pygame
import random
import numpy as np
import sys
from collections import deque
import math
import atexit
import time

# Initialize pygame
pygame.init()

# Constants
SCREEN_WIDTH = 1200
SCREEN_HEIGHT = 800
CELL_SIZE = 40
GRID_WIDTH = 20
GRID_HEIGHT = 15
PANEL_WIDTH = 400
FPS = 60

# Colors
BACKGROUND = (30, 30, 50)
GRID_LINES = (60, 60, 80)
WALL_COLOR = (50, 50, 70)
PATH_COLOR = (35, 35, 55)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
PANEL_BG = (25, 25, 40)
BUTTON_COLOR = (70, 130, 200)
BUTTON_HOVER = (90, 150, 220)
Q_VALUE_COLORS = [
    (180, 60, 60),    # Low values (red)
    (220, 150, 60),   # Medium values (orange)
    (60, 180, 80)     # High values (green)
]

# Create screen
screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Reinforcement Learning Maze Environment")
clock = pygame.time.Clock()

# Fonts
font_large = pygame.font.SysFont("Arial", 36, bold=True)
font_medium = pygame.font.SysFont("Arial", 24)
font_small = pygame.font.SysFont("Arial", 18)

# Register cleanup function
def cleanup():
    pygame.quit()

atexit.register(cleanup)

class Maze:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.grid = np.zeros((height, width), dtype=int)  # 0 = path, 1 = wall
        self.start_pos = (1, 1)
        self.goal_pos = (width - 2, height - 2)
        self.hazards = []
        self.generate_maze()
        
    def generate_maze(self):
        # Initialize with walls
        self.grid.fill(1)
        
        # Create a depth-first search maze
        stack = [self.start_pos]
        self.grid[self.start_pos[1]][self.start_pos[0]] = 0
        
        # Possible directions
        directions = [(0, -2), (2, 0), (0, 2), (-2, 0)]  # Up, right, down, left
        
        while stack:
            current = stack[-1]
            x, y = current
            
            # Find unvisited neighbors
            neighbors = []
            for dx, dy in directions:
                nx, ny = x + dx, y + dy
                if 0 < nx < self.width - 1 and 0 < ny < self.height - 1 and self.grid[ny][nx] == 1:
                    neighbors.append((nx, ny, dx, dy))
            
            if neighbors:
                # Choose a random neighbor
                nx, ny, dx, dy = random.choice(neighbors)
                
                # Remove wall between current and neighbor
                self.grid[y + dy//2][x + dx//2] = 0
                self.grid[ny][nx] = 0
                
                # Add to stack
                stack.append((nx, ny))
            else:
                # Backtrack
                stack.pop()
        
        # Ensure there's a path to the goal
        self.create_path_to_goal()
        
        # Add hazards (10% of path cells)
        path_cells = [(x, y) for y in range(self.height) for x in range(self.width) 
                      if self.grid[y][x] == 0 and (x, y) != self.start_pos and (x, y) != self.goal_pos]
        
        num_hazards = max(1, int(len(path_cells) * 0.1))
        self.hazards = random.sample(path_cells, num_hazards)
    
    def create_path_to_goal(self):
        # Use BFS to find a path from start to goal
        queue = deque([self.start_pos])
        visited = {self.start_pos: None}
        
        while queue:
            x, y = queue.popleft()
            
            if (x, y) == self.goal_pos:
                break
                
            for dx, dy in [(0, -1), (1, 0), (0, 1), (-1, 0)]:
                nx, ny = x + dx, y + dy
                if (0 <= nx < self.width and 0 <= ny < self.height and 
                    self.grid[ny][nx] == 0 and (nx, ny) not in visited):
                    queue.append((nx, ny))
                    visited[(nx, ny)] = (x, y)
        
        # If no path found, create one
        if self.goal_pos not in visited:
            x, y = self.goal_pos
            path = []
            
            # Create a path backwards from goal to start
            while (x, y) != self.start_pos:
                path.append((x, y))
                # Move toward start (simplified)
                if x > self.start_pos[0]:
                    x -= 1
                elif x < self.start_pos[0]:
                    x += 1
                elif y > self.start_pos[1]:
                    y -= 1
                elif y < self.start_pos[1]:
                    y += 1
                
                # Remove wall at this position
                if 0 <= x < self.width and 0 <= y < self.height:
                    self.grid[y][x] = 0
            
            # Add the path to the maze
            for pos in path:
                self.grid[pos[1]][pos[0]] = 0
    
    def is_wall(self, x, y):
        return self.grid[y][x] == 1
    
    def is_hazard(self, x, y):
        return (x, y) in self.hazards

class Player:
    def __init__(self, maze):
        self.maze = maze
        self.reset()
        self.lives = 30
        self.speed = 0.2
        self.radius = CELL_SIZE // 3
        self.trail = []
        
    def reset(self):
        self.x, self.y = self.maze.start_pos
        self.px, self.py = self.x, self.y
        self.target_x, self.target_y = self.x, self.y
        self.moving = False
        self.trail = []
        
    def move(self, dx, dy):
        if self.moving:
            return
            
        new_x, new_y = self.x + dx, self.y + dy
        
        # Check boundaries
        if (0 <= new_x < self.maze.width and 0 <= new_y < self.maze.height and 
            not self.maze.is_wall(new_x, new_y)):
            self.px, self.py = self.x, self.y
            self.target_x, self.target_y = new_x, new_y
            self.moving = True
            
            # Add to trail with current timestamp
            self.trail.append((self.x, self.y, time.time()))
    
    def update(self):
        if self.moving:
            # Move toward target
            dx = self.target_x - self.x
            dy = self.target_y - self.y
            dist = math.sqrt(dx*dx + dy*dy)
            
            if dist < self.speed:
                self.x, self.y = self.target_x, self.target_y
                self.moving = False
            else:
                self.x += dx * self.speed
                self.y += dy * self.speed
        
        # Remove trail points older than 1 second
        current_time = time.time()
        self.trail = [(x, y, t) for (x, y, t) in self.trail if current_time - t < 1.0]
    
    def get_position(self):
        return (int(self.x), int(self.y))
    
    def lose_life(self):
        self.lives -= 1
        self.reset()
        return self.lives <= 0
    
    def draw(self, screen, offset_x, offset_y):
        # Draw trail
        current_time = time.time()
        for i, (tx, ty, t) in enumerate(self.trail):
            # Calculate alpha based on remaining time (fade out over 1 second)
            alpha = int(255 * (1 - (current_time - t)))
            if alpha <= 0:
                continue
                
            # Create a surface with per-pixel alpha
            trail_surface = pygame.Surface((CELL_SIZE, CELL_SIZE), pygame.SRCALPHA)
            radius = int(self.radius * (1 - (current_time - t)))
            pygame.draw.circle(trail_surface, (70, 130, 180, alpha), 
                              (CELL_SIZE // 2, CELL_SIZE // 2), 
                              radius)
            screen.blit(trail_surface, (offset_x + tx * CELL_SIZE, offset_y + ty * CELL_SIZE))
        
        # Draw player
        pygame.draw.circle(screen, PLAYER_COLOR, 
                          (offset_x + (self.x + 0.5) * CELL_SIZE, offset_y + (self.y + 0.5) * CELL_SIZE), 
                          self.radius)
        
        # Draw player direction indicator
        if self.moving:
            dx = self.target_x - self.px
            dy = self.target_y - self.py
            if dx != 0 or dy != 0:
                angle = math.atan2(dy, dx)
                end_x = (self.x + 0.5 + 0.6 * math.cos(angle)) * CELL_SIZE
                end_y = (self.y + 0.5 + 0.6 * math.sin(angle)) * CELL_SIZE
                pygame.draw.line(screen, (255, 255, 255), 
                                (offset_x + (self.x + 0.5) * CELL_SIZE, offset_y + (self.y + 0.5) * CELL_SIZE),
                                (offset_x + end_x, offset_y + end_y), 3)

class QLearningAgent:
    def __init__(self, maze, player):
        self.maze = maze
        self.player = player
        self.q_table = {}
        self.learning_rate = 0.1
        self.discount_factor = 0.95
        self.exploration_rate = 0.3
        self.state = None
        self.action = None
        
    def get_state(self):
        return self.player.get_position()
    
    def get_q_value(self, state, action):
        if state not in self.q_table:
            self.q_table[state] = [0.0, 0.0, 0.0, 0.0]  # Up, Right, Down, Left
        return self.q_table[state][action]
    
    def choose_action(self, state):
        # Exploration vs exploitation
        if random.random() < self.exploration_rate:
            return random.randint(0, 3)  # Random action
        else:
            # Choose best action
            q_values = [self.get_q_value(state, a) for a in range(4)]
            max_q = max(q_values)
            # If multiple actions have the same max q value, choose randomly among them
            actions = [a for a, q in enumerate(q_values) if q == max_q]
            return random.choice(actions)
    
    def update_q_value(self, state, action, reward, next_state):
        current_q = self.get_q_value(state, action)
        next_max_q = max([self.get_q_value(next_state, a) for a in range(4)])
        
        # Q-learning formula
        new_q = current_q + self.learning_rate * (reward + self.discount_factor * next_max_q - current_q)
        self.q_table[state][action] = new_q
    
    def start_episode(self):
        self.state = self.get_state()
        self.action = self.choose_action(self.state)
        return self.action
    
    def step(self, reward, next_state):
        # Update Q-value for previous state-action pair
        self.update_q_value(self.state, self.action, reward, next_state)
        
        # Choose next action
        self.state = next_state
        self.action = self.choose_action(self.state)
        return self.action
    
    def get_q_values(self):
        return self.q_table

class Button:
    def __init__(self, x, y, width, height, text, action=None):
        self.rect = pygame.Rect(x, y, width, height)
        self.text = text
        self.action = action
        self.hovered = False
        
    def draw(self, screen):
        color = BUTTON_HOVER if self.hovered else BUTTON_COLOR
        pygame.draw.rect(screen, color, self.rect, border_radius=8)
        pygame.draw.rect(screen, (100, 160, 230), self.rect, 2, border_radius=8)
        
        text_surf = font_medium.render(self.text, True, TEXT_COLOR)
        text_rect = text_surf.get_rect(center=self.rect.center)
        screen.blit(text_surf, text_rect)
        
    def check_hover(self, pos):
        self.hovered = self.rect.collidepoint(pos)
        
    def handle_event(self, event):
        if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            if self.hovered and self.action:
                return self.action()
        return None

class Game:
    def __init__(self):
        self.maze = Maze(GRID_WIDTH, GRID_HEIGHT)
        self.player = Player(self.maze)
        self.agent = QLearningAgent(self.maze, self.player)
        self.game_state = "PLAYING"  # PLAYING, DIED, WON
        self.offset_x = 50
        self.offset_y = 50
        self.manual_control = True
        self.test_mode = False  # New flag for test mode
        self.episode_count = 0
        self.steps = 0
        self.episode_reward = 0
        self.total_reward = 0
        self.convergence_data = []
        self.running = True
        self.start_time = time.time()
        self.simulation_speed = 1  # 1x normal speed
        self.speed_multipliers = [1, 100, 1000, 10000, 100000]  # Available speed multipliers
        self.current_speed_index = 0  # Start with normal speed
        
        # Create buttons
        button_width = 150
        button_height = 40
        button_spacing = 20
        panel_center = SCREEN_WIDTH - PANEL_WIDTH // 2
        self.buttons = [
            Button(panel_center - button_width//2, 500, button_width, button_height, "Manual Control", self.set_manual),
            Button(panel_center - button_width//2, 500 + button_height + button_spacing, button_width, button_height, "Auto Training", self.set_auto),
            Button(panel_center - button_width//2, 500 + 2*(button_height + button_spacing), button_width, button_height, "Test Mode", self.set_test),
            Button(panel_center - button_width//2, 500 + 3*(button_height + button_spacing), button_width, button_height, "Reset Game", self.reset_game),
            Button(panel_center - button_width//2, 500 + 4*(button_height + button_spacing), button_width, button_height, "Quit", self.quit_game),
            Button(panel_center - button_width//2, 500 + 5*(button_height + button_spacing), button_width, button_height, f"Speed: {self.simulation_speed}x", self.toggle_speed)
        ]
    
    def set_manual(self):
        self.manual_control = True
        self.test_mode = False
        return True
    
    def set_auto(self):
        self.manual_control = False
        self.test_mode = False
        return True
    
    def set_test(self):
        self.manual_control = False
        self.test_mode = True
        return True
    
    def reset_game(self):
        self.maze = Maze(GRID_WIDTH, GRID_HEIGHT)
        self.player = Player(self.maze)
        self.agent = QLearningAgent(self.maze, self.player)
        self.game_state = "PLAYING"
        self.episode_count = 0
        self.steps = 0
        self.episode_reward = 0
        self.total_reward = 0
        self.convergence_data = []
        self.start_time = time.time()
        return True
    
    def quit_game(self):
        self.running = False
        return True
    
    def toggle_speed(self):
        self.current_speed_index = (self.current_speed_index + 1) % len(self.speed_multipliers)
        self.simulation_speed = self.speed_multipliers[self.current_speed_index]
        self.buttons[5].text = f"Speed: {self.simulation_speed}x"
        return True
    
    def get_elapsed_time(self):
        return int(time.time() - self.start_time)
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.running = False
                return False
                
            # Handle button events
            for button in self.buttons:
                result = button.handle_event(event)
                if result is not None and button.text.startswith("Speed:"):
                    # Update speed button text
                    button.text = f"Speed: {self.simulation_speed}x"
            
            # Handle keyboard events
            if event.type == pygame.KEYDOWN and self.game_state == "PLAYING":
                if event.key == pygame.K_UP and self.manual_control:
                    self.player.move(0, -1)
                elif event.key == pygame.K_RIGHT and self.manual_control:
                    self.player.move(1, 0)
                elif event.key == pygame.K_DOWN and self.manual_control:
                    self.player.move(0, 1)
                elif event.key == pygame.K_LEFT and self.manual_control:
                    self.player.move(-1, 0)
                elif event.key == pygame.K_r:
                    self.player.reset()
                elif event.key == pygame.K_SPACE:
                    self.manual_control = not self.manual_control
                elif event.key == pygame.K_ESCAPE:
                    self.running = False
                    return False
                elif event.key == pygame.K_PLUS or event.key == pygame.K_EQUALS:
                    self.toggle_speed()
                elif event.key == pygame.K_MINUS:
                    self.current_speed_index = (self.current_speed_index - 1) % len(self.speed_multipliers)
                    self.simulation_speed = self.speed_multipliers[self.current_speed_index]
                    self.buttons[5].text = f"Speed: {self.simulation_speed}x"
                    
            # Handle mouse events for buttons
            if event.type == pygame.MOUSEMOTION:
                for button in self.buttons:
                    button.check_hover(event.pos)
        
        return True
    
    def update(self):
        if self.game_state != "PLAYING":
            # Automatically continue to next episode
            if self.game_state == "WON" or (self.game_state == "DIED" and not self.test_mode):
                self.player.reset()
                self.game_state = "PLAYING"
            return
            
        self.player.update()
        player_pos = self.player.get_position()
        
        # Check for hazard collision
        if self.maze.is_hazard(player_pos[0], player_pos[1]):
            self.episode_reward -= 50
            self.total_reward -= 50
            
            # In auto training mode, reset immediately with penalty
            if not self.test_mode and not self.manual_control:
                self.player.reset()
            # In test mode or manual control, lose a life
            elif self.test_mode or self.manual_control:
                dead = self.player.lose_life()
                if dead:
                    self.game_state = "DIED"
        
        # Check for goal reached
        if player_pos == self.maze.goal_pos:
            self.episode_count += 1
            self.episode_reward += 100
            self.total_reward += 100
            self.convergence_data.append(self.steps)
            self.steps = 0
            self.game_state = "WON"
            
            # In auto training mode, reset immediately
            if not self.test_mode:
                self.player.reset()
                self.game_state = "PLAYING"
        
        # Auto control (Q-learning)
        if not self.manual_control and not self.player.moving:
            # Get current state
            current_state = self.agent.get_state()
            
            # Choose and perform action
            if self.agent.state is None:
                action = self.agent.start_episode()
            else:
                # Calculate reward
                reward = -0.1  # Small penalty for each step
                self.episode_reward += reward
                self.total_reward += reward
                
                # Update Q-value and get next action
                action = self.agent.step(reward, current_state)
            
            # Perform action
            if action == 0:  # Up
                self.player.move(0, -1)
            elif action == 1:  # Right
                self.player.move(1, 0)
            elif action == 2:  # Down
                self.player.move(0, 1)
            elif action == 3:  # Left
                self.player.move(-1, 0)
            
            self.steps += 1
    
    def draw(self):
        screen.fill(BACKGROUND)
        
        # Draw maze and player
        self.draw_maze()
        self.player.draw(screen, self.offset_x, self.offset_y)
        
        # Draw right panel
        self.draw_panel()
        
        # Draw game state messages
        if self.game_state == "DIED":
            self.draw_message("YOU DIED!", HAZARD_COLOR)
        elif self.game_state == "WON":
            self.draw_message("LEVEL COMPLETE!", GOAL_COLOR)
        
        pygame.display.flip()
        # Adjust FPS based on simulation speed
        clock.tick(FPS * self.simulation_speed)
    
    def draw_maze(self):
        # Draw maze grid
        for y in range(self.maze.height):
            for x in range(self.maze.width):
                rect = pygame.Rect(
                    self.offset_x + x * CELL_SIZE,
                    self.offset_y + y * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                if self.maze.is_wall(x, y):
                    pygame.draw.rect(screen, WALL_COLOR, rect)
                else:
                    pygame.draw.rect(screen, PATH_COLOR, rect)
                
                # Draw grid lines
                pygame.draw.rect(screen, GRID_LINES, rect, 1)
        
        # Draw start and goal
        start_rect = pygame.Rect(
            self.offset_x + self.maze.start_pos[0] * CELL_SIZE,
            self.offset_y + self.maze.start_pos[1] * CELL_SIZE,
            CELL_SIZE, CELL_SIZE
        )
        pygame.draw.rect(screen, (70, 100, 150), start_rect)
        
        goal_rect = pygame.Rect(
            self.offset_x + self.maze.goal_pos[0] * CELL_SIZE,
            self.offset_y + self.maze.goal_pos[1] * CELL_SIZE,
            CELL_SIZE, CELL_SIZE
        )
        pygame.draw.rect(screen, GOAL_COLOR, goal_rect)
        
        # Draw hazards
        for hazard in self.maze.hazards:
            hazard_rect = pygame.Rect(
                self.offset_x + hazard[0] * CELL_SIZE,
                self.offset_y + hazard[1] * CELL_SIZE,
                CELL_SIZE, CELL_SIZE
            )
            pygame.draw.rect(screen, HAZARD_COLOR, hazard_rect, border_radius=8)
    
    def draw_panel(self):
        # Draw panel background
        panel_rect = pygame.Rect(SCREEN_WIDTH - PANEL_WIDTH, 0, PANEL_WIDTH, SCREEN_HEIGHT)
        pygame.draw.rect(screen, PANEL_BG, panel_rect)
        pygame.draw.line(screen, (60, 60, 90), (SCREEN_WIDTH - PANEL_WIDTH, 0), 
                         (SCREEN_WIDTH - PANEL_WIDTH, SCREEN_HEIGHT), 3)
        
        # Draw title
        title = font_large.render("RL MAZE", True, TEXT_COLOR)
        screen.blit(title, (SCREEN_WIDTH - PANEL_WIDTH + 20, 20))
        
        # Draw game info
        info_y = 80
        info = [
            f"Lives: {self.player.lives}",
            f"Episodes: {self.episode_count}",
            f"Steps: {self.steps}",
            f"Total Reward: {self.total_reward:.1f}",
            f"Mode: {'Manual' if self.manual_control else 'Test' if self.test_mode else 'Auto Training'}",
            f"Time: {self.get_elapsed_time()}s",
            f"Speed: {self.simulation_speed}x"
        ]
        
        for text in info:
            text_surf = font_medium.render(text, True, TEXT_COLOR)
            screen.blit(text_surf, (SCREEN_WIDTH - PANEL_WIDTH + 20, info_y))
            info_y += 40
        
        # Draw Q-learning parameters
        params_y = info_y + 20
        params = [
            "Q-Learning Parameters:",
            f"Learning Rate: {self.agent.learning_rate}",
            f"Discount Factor: {self.agent.discount_factor}",
            f"Exploration Rate: {self.agent.exploration_rate}"
        ]
        
        for text in params:
            text_surf = font_small.render(text, True, TEXT_COLOR)
            screen.blit(text_surf, (SCREEN_WIDTH - PANEL_WIDTH + 20, params_y))
            params_y += 30
        
        # Draw convergence graph
        if self.convergence_data:
            graph_title = font_medium.render("Convergence Progress", True, TEXT_COLOR)
            screen.blit(graph_title, (SCREEN_WIDTH - PANEL_WIDTH + 20, params_y))
            params_y += 40
            
            # Draw graph background
            graph_rect = pygame.Rect(SCREEN_WIDTH - PANEL_WIDTH + 20, params_y, PANEL_WIDTH - 60, 150)
            pygame.draw.rect(screen, (40, 40, 60), graph_rect)
            pygame.draw.rect(screen, (80, 80, 120), graph_rect, 1)
            
            # Draw data points
            if len(self.convergence_data) > 1:
                max_steps = max(self.convergence_data)
                min_steps = min(self.convergence_data)
                range_steps = max_steps - min_steps if max_steps > min_steps else 1
                
                x_step = (graph_rect.width - 20) / (len(self.convergence_data) - 1)
                points = []
                
                for i, steps in enumerate(self.convergence_data):
                    x = graph_rect.x + 10 + i * x_step
                    y = graph_rect.y + 10 + (steps - min_steps) * 130 / range_steps
                    points.append((x, y))
                
                if len(points) > 1:
                    pygame.draw.lines(screen, (70, 180, 220), False, points, 2)
                
                # Draw labels
                min_text = font_small.render(f"{min_steps}", True, TEXT_COLOR)
                max_text = font_small.render(f"{max_steps}", True, TEXT_COLOR)
                screen.blit(min_text, (graph_rect.right - 30, graph_rect.bottom - 15))
                screen.blit(max_text, (graph_rect.right - 30, graph_rect.top + 5))
        
        # Draw buttons
        for button in self.buttons:
            button.draw(screen)
    
    def draw_message(self, text, color):
        overlay = pygame.Surface((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.SRCALPHA)
        overlay.fill((0, 0, 0, 180))
        screen.blit(overlay, (0, 0))
        
        text_surf = font_large.render(text, True, color)
        text_rect = text_surf.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 - 50))
        screen.blit(text_surf, text_rect)
        
        if self.game_state == "DIED":
            hint = font_medium.render("Press 'Reset Game' to play again", True, TEXT_COLOR)
        else:
            hint = font_medium.render("Press any key to continue", True, TEXT_COLOR)
        
        hint_rect = hint.get_rect(center=(SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + 20))
        screen.blit(hint, hint_rect)
    
    def run(self):
        self.running = True
        while self.running:
            if not self.handle_events():
                self.running = False
                break
            self.update()
            self.draw()
        
        pygame.quit()

# Start the game
if __name__ == "__main__":
    game = Game()
    game.run()